# 21. Annotation Mask Test (Pipeline 02)

- Goal: load, validate, and apply frame-level annotation metadata without changing coordinates.
- Docs: `docs_eng/pipeline/02_annotation.md` / `docs/pipeline/02_annotation.md`
- Inputs: Default pose CSV and optional annotation CSV paths from the input cell.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Fallback behavior, annotation validation, applied segment summary, and overlap rejection.


In [ ]:
from pathlib import Path

import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv, validate_annotation
from movement.io import load_pose_csv


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"

pose_df = load_pose_csv(csv_path)

pd.DataFrame([
    {"input": "pose_csv", "path": csv_path.relative_to(PROJECT_ROOT).as_posix(), "exists": csv_path.exists()},
    {"input": "annotation_csv", "path": ann_path.relative_to(PROJECT_ROOT).as_posix(), "exists": ann_path.exists()},
    {"input": "pose_dataframe", "path": "in_memory", "exists": True, "frames": len(pose_df), "columns": pose_df.shape[1]},
])


## Fallback


In [ ]:
annotated_fallback, report_fallback = apply_annotation(pose_df, ann_df=None)

fallback_summary = pd.DataFrame([{
    "annotation_provided": report_fallback["annotation_provided"],
    "num_total_frames": report_fallback["num_total_frames"],
    "num_analysis_frames": report_fallback["num_analysis_frames"],
    "segment_type": ", ".join(map(str, annotated_fallback["segment_type"].unique())),
    "execution_pattern": ", ".join(map(str, annotated_fallback["execution_pattern"].unique())),
}])
display(fallback_summary)

assert report_fallback["annotation_provided"] is False
assert report_fallback["num_analysis_frames"] == len(pose_df)
assert annotated_fallback["segment_type"].eq("full_sequence").all()


## Annotation File and Validation


In [ ]:
ann_df = load_annotation_csv(ann_path)

preview_cols = [
    "segment_type", "set_id", "rep_id", "start_frame", "end_frame",
    "use_for_analysis", "exercise_id", "execution_pattern", "starting_side",
]
display(ann_df[[col for col in preview_cols if col in ann_df.columns]])

val_report = validate_annotation(ann_df, pose_df)
validation_summary = pd.DataFrame([{
    "passed": val_report["passed"],
    "missing_required_columns": len(val_report["missing_required_columns"]),
    "invalid_range_rows": len(val_report["invalid_range_rows"]),
    "out_of_bounds_rows": len(val_report["out_of_bounds_rows"]),
    "overlapping_pairs": len(val_report["overlapping_pairs"]),
    "unknown_segment_types": len(val_report["unknown_segment_types"]),
}])
display(validation_summary)

assert val_report["passed"] is True


## Apply Annotation


In [ ]:
annotated_df, ann_report = apply_annotation(pose_df, ann_df)

report_summary = pd.DataFrame([{
    "annotation_provided": ann_report["annotation_provided"],
    "num_total_frames": ann_report["num_total_frames"],
    "num_analysis_frames": ann_report["num_analysis_frames"],
    "num_excluded_frames": ann_report["num_excluded_frames"],
    "num_annotated_rows": ann_report["num_annotated_rows"],
    "num_sets": ann_report["num_sets"],
    "num_reps": ann_report["num_reps"],
}])
display(report_summary)

display(pd.DataFrame([ann_report["performance_provenance"]["summary"]]))

segment_summary = (
    annotated_df.groupby("segment_type", dropna=False)
    .agg(frames=("frame", "count"), analysis_frames=("use_for_analysis", "sum"))
    .reset_index()
)
display(segment_summary)

rep_summary = (
    annotated_df[annotated_df["segment_type"] == "rep"]
    .groupby(["set_id", "rep_id"], dropna=False)
    .agg(start=("frame", "min"), end=("frame", "max"), frames=("frame", "count"))
    .reset_index()
)
display(rep_summary)

assert ann_report["num_analysis_frames"] + ann_report["num_excluded_frames"] == ann_report["num_total_frames"]
assert annotated_df["frame"].equals(pose_df["frame"])


## Context Columns


In [ ]:
context_rows = []
for label, frame_df in [
    ("annotated", annotated_df),
    ("fallback", annotated_fallback),
]:
    context_rows.append({
        "source": label,
        "exercise_id": frame_df["exercise_id"].dropna().unique().tolist(),
        "execution_pattern": frame_df["execution_pattern"].dropna().unique().tolist(),
        "starting_side": frame_df["starting_side"].dropna().unique().tolist(),
    })
display(pd.DataFrame(context_rows))

segment_context = (
    annotated_df[["segment_type", "exercise_id", "execution_pattern"]]
    .dropna(subset=["exercise_id", "execution_pattern"])
    .groupby(["segment_type", "exercise_id", "execution_pattern"], dropna=False)
    .size()
    .reset_index(name="frames")
)
display(segment_context)


## Overlap Guard


In [ ]:
bad_ann = pd.DataFrame({
    "segment_type": ["rep", "rep"],
    "set_id": pd.array([1, 1], dtype="Int64"),
    "rep_id": pd.array([1, 2], dtype="Int64"),
    "start_frame": [10, 50],
    "end_frame": [70, 90],
    "use_for_analysis": [True, True],
})

overlap_report = validate_annotation(bad_ann, pose_df)
try:
    apply_annotation(pose_df, bad_ann)
    overlap_error_raised = False
except ValueError:
    overlap_error_raised = True

display(pd.DataFrame([{
    "validation_passed": overlap_report["passed"],
    "overlapping_pairs": overlap_report["overlapping_pairs"],
    "apply_raised_value_error": overlap_error_raised,
}]))

assert overlap_report["passed"] is False
assert overlap_report["overlapping_pairs"]
assert overlap_error_raised is True
